In [1]:
import os; os.environ['WORKDIR'] = "/Users/choij/workspace/ChargedHiggsAnalysis"
import sys; sys.path.insert(0, os.environ['WORKDIR'])

from ROOT import TFile
from math import sqrt, log
from libPython.DataDriven import Conversion

Welcome to JupyROOT 6.26/06


In [18]:
# signal
MASSPOINT = "MHc-130_MA-90"
BACKGROUND = "ttX"
# backgrounds
DataStream = "DoubleMuon"   # for fake
Conv = ["DYJets", "ZGToLLG"]
VV = ["WZTo3LNu_amcatnlo", "ZZTo4L_powheg"]
#VV = ["WZTo3LNu_amcatnlo"]
ttX = ["ttWToLNu", "ttZToLLNuNu", "ttHToNonbb", "tZq", "tHq"]
Rare = ["WWW", "WWZ", "WZZ", "ZZZ", "WWG", "TTG", "TTTT", "VBF_HToZZTo4L", "GluGluHToZZTo4L"]
Total = Conv + VV + ttX + Rare

ERA = "2018"

Systematics = ["Central",
               ["L1PrefireUp", "L1PrefireDown"],
               ["PileUpCorrUp", "PileUpCorrDown"],
               ["MuonMomentumShiftUp", "MuonMomentumShiftDown"],
               ["JetEnShiftUp", "JetEnShiftDown"],
               ["JetResShiftUp", "JetResShiftDown"],
               ["MuonIDSFUp", "MuonIDSFDown"],
               ["DblMuonTrigSFUp", "DblMuonTrigSFDown"]
               ]

In [19]:
# get histograms
# signal
f = TFile.Open(f"../ROOT/Skim3Mu__/2018/TTToHcToWAToMuMu_{MASSPOINT}.root")
central, *systs = Systematics
h_sig = f.Get(f"3Mu/SignalRegion/Central/Incl/Outputs/{MASSPOINT}/score_vs_{BACKGROUND}_mACand")
h_sig.SetDirectory(0)
h_systs = []
for syst in systs:
    h_up = f.Get(f"3Mu/SignalRegion/{syst[0]}/Incl/Outputs/{MASSPOINT}/score_vs_{BACKGROUND}_mACand")
    h_up.SetDirectory(0)
    h_down = f.Get(f"3Mu/SignalRegion/{syst[1]}/Incl/Outputs/{MASSPOINT}/score_vs_{BACKGROUND}_mACand")
    h_down.SetDirectory(0)
    h_systs.append([h_up, h_down])
f.Close()

for bin in range(h_sig.GetNcells()):
    this_value, this_error = h_sig.GetBinContent(bin), h_sig.GetBinError(bin)
    this_error = pow(this_error, 2)
    for syst in h_systs:
        this_syst_up = syst[0].GetBinContent(bin) - this_value
        this_syst_down = syst[1].GetBinContent(bin) - this_value
        this_syst = max(abs(this_syst_up), abs(this_syst_down))
        this_error += pow(this_syst, 2)
    this_error = sqrt(this_error)
    h_sig.SetBinError(bin, this_error)

In [20]:
# get histograms
# fake
f = TFile.Open(f"../ROOT/Skim3Mu__/{ERA}/{DataStream}.root")
h_fake = f.Get(f"3Mu/SignalRegion/Central/Incl/Outputs/{MASSPOINT}/score_vs_{BACKGROUND}_mACand")
h_fake.SetDirectory(0)
h_fake_up = f.Get(f"3Mu/SignalRegion/NonpromptUp/Incl/Outputs/{MASSPOINT}/score_vs_{BACKGROUND}_mACand")
h_fake_up.SetDirectory(0)
h_fake_down = f.Get(f"3Mu/SignalRegion/NonpromptDown/Incl/Outputs/{MASSPOINT}/score_vs_{BACKGROUND}_mACand")
h_fake_down.SetDirectory(0)
f.Close()

for bin in range(h_fake.GetNcells()):
    this_value, this_error = h_fake.GetBinContent(bin), h_fake.GetBinError(bin)
    this_error = pow(this_error, 2)
    this_syst_up = h_fake_up.GetBinContent(bin) - this_value
    this_syst_down = h_fake_down.GetBinContent(bin) - this_value
    this_syst = max(abs(this_syst_up), abs(this_syst_down))
    this_error += pow(this_syst, 2)
    this_error = sqrt(this_error)
    h_fake.SetBinError(bin, this_error)

In [21]:
# other backgrounds
MCcoll = {}
ConvSF = Conversion(era=ERA)
for sample in Total:
    # conversion
    if sample in Conv:
        measure = "DYJets" if sample == "DYJets" else "ZGamma"
        f = TFile.Open(f"../ROOT/Skim3Mu__/{ERA}/{sample}.root")
        h_cent = f.Get(f"3Mu/SignalRegion/Central/{measure}/Outputs/{MASSPOINT}/score_vs_{BACKGROUND}_mACand")
        if not h_cent: continue
        h_cent.SetDirectory(0)
        h_up = h_cent.Clone("conv_up")
        h_down = h_cent.Clone("conv_down")
        
        # scale and set systematics
        h_cent.Scale(ConvSF.getScale(measure))
        h_up.Scale(ConvSF.getScale(measure, 1))
        h_down.Scale(ConvSF.getScale(measure, -1))
        for bin in range(h_cent.GetNcells()):
            this_value = h_cent.GetBinContent(bin)
            this_syst_up = h_up.GetBinContent(bin) - this_value
            this_syst_down = h_down.GetBinContent(bin) - this_value
            this_syst = max(abs(this_syst_up), abs(this_syst_down))
            h_cent.SetBinError(bin, this_syst)
    else:
        central, *systs = Systematics
        f = TFile.Open(f"../ROOT/Skim3Mu__/{ERA}/{sample}.root")
        h_cent = f.Get(f"3Mu/SignalRegion/Central/Incl/Outputs/{MASSPOINT}/score_vs_{BACKGROUND}_mACand")
        if not h_cent:
            continue
        h_cent.SetDirectory(0)
        h_systs = []
        for syst in systs:
            h_up = f.Get(f"3Mu/SignalRegion/{syst[0]}/Incl/Outputs/{MASSPOINT}/score_vs_{BACKGROUND}_mACand")
            h_up.SetDirectory(0)
            h_down = f.Get(f"3Mu/SignalRegion/{syst[1]}/Incl/Outputs/{MASSPOINT}/score_vs_{BACKGROUND}_mACand")
            h_down.SetDirectory(0)
            h_systs.append([h_up, h_down])
        f.Close()

        for bin in range(h_cent.GetNcells()):
            this_value, this_error = h_cent.GetBinContent(bin), h_cent.GetBinError(bin)
            this_error = pow(this_error, 2)
            for syst in h_systs:
                this_syst_up = syst[0].GetBinContent(bin) - this_value
                this_syst_down = syst[1].GetBinContent(bin) - this_value
                this_syst = max(abs(this_syst_up), abs(this_syst_down))
                this_error += pow(this_syst, 2)
            this_error = sqrt(this_error)
            h_cent.SetBinError(bin, this_error)
    if sample == "ZZTo4L_powheg":
        h_cent.Scale(186./19.)
    MCcoll[sample] = h_cent

In [22]:
# optimization
def getRates(firstxbin, lastxbin, firstybin, lastybin):
    rateSig = h_sig.Integral(firstxbin, lastxbin, firstybin, lastybin)
    rateFake = h_fake.Integral(firstxbin, lastxbin, firstybin, lastybin)
    rateConv = 0.
    rateVV = 0.
    rateTTX = 0.
    rateRare = 0.
    for sample in Conv:
        rateConv += MCcoll[sample].Integral(firstxbin, lastxbin, firstybin, lastybin)
    for sample in VV:
        rateVV += MCcoll[sample].Integral(firstxbin, lastxbin, firstybin, lastybin)
    for sample in ttX:
        rateTTX += MCcoll[sample].Integral(firstxbin, lastxbin, firstybin, lastybin)
    for sample in Rare: 
        rateRare += MCcoll[sample].Integral(firstxbin, lastxbin, firstybin, lastybin)
    return (rateSig, rateFake, rateConv, rateVV, rateTTX, rateRare)

def getRatesBySample(sample, firstxbin, lastxbin, firstybin, lastybin):
    rate = MCcoll[sample].Integral(firstxbin, lastxbin, firstybin, lastybin)
    return rate

def getTestStat(rateSig, rateBkg):
    return sqrt(2*((rateSig+rateBkg)*log(1+rateSig/rateBkg) - rateSig)) 

def optimize(mA, massWindow, arrScore):
    bestTestStat = 0.
    bestCut = 0.
    # 1 GeV step
    firstybin = h_sig.GetYaxis().FindBin(mA-massWindow)
    lastybin = h_sig.GetYaxis().FindBin(mA+massWindow)
    for score in arrScore:
        firstxbin = h_sig.GetXaxis().FindBin(score)
        lastxbin = h_sig.GetXaxis().FindBin(1.)

        rateSig, rateFake, rateConv, rateVV, rateTTX, rateRare = getRates(firstxbin, lastxbin, firstybin, lastybin)
        rateBkg = rateFake+rateConv+rateVV+rateTTX+rateRare
        testStat = getTestStat(rateSig, rateBkg)
        
        if testStat > bestTestStat:
            bestTestStat = testStat
            bestCut = score
    return (bestCut, bestTestStat)

In [23]:
import numpy as np
mA = 90
window = 2
arrScore = np.linspace(0., 1., 101)[:-1]
bestCut, bestTestStat = optimize(mA, window, arrScore=arrScore)
print(bestCut, bestTestStat)

0.01 5.09704883218249


In [24]:
firstxbin = h_sig.GetXaxis().FindBin(bestCut)
lastxbin = h_sig.GetXaxis().FindBin(1.)
firstybin = h_sig.GetYaxis().FindBin(mA-window)
lastybin = h_sig.GetYaxis().FindBin(mA+window)

rateSig, rateFake, rateConv, rateVV, rateTTX, rateRare = getRates(firstxbin, lastxbin, firstybin, lastybin)
rateBkg = rateFake+rateConv+rateVV+rateTTX+rateRare
testStat = getTestStat(rateSig, rateBkg)
print(f"rateSig: {rateSig}")
print(f"rateFake: {rateFake}")
print(f"rateConv: {rateConv}")
print(f"rateVV: {rateVV}")
print(f"rateTTX: {rateTTX}")
print(f"rateRare: {rateRare}")
print(f"rateBkg: {rateBkg}")
print(f"testStat: {testStat}")

print(getRatesBySample("ttZToLLNuNu", firstxbin, lastxbin, firstybin, lastybin))

rateSig: 98.0460566085346
rateFake: 223.0
rateConv: -1.2495581113758063
rateVV: 48.9785117435963
rateTTX: 65.43937961650255
rateRare: 2.561951396292324
rateBkg: 338.7302846450154
testStat: 5.09704883218249
42.265439206495245


In [25]:
firstxbin = h_sig.GetXaxis().FindBin(0.)
lastxbin = h_sig.GetXaxis().FindBin(1.)
firstybin = h_sig.GetYaxis().FindBin(mA-window)
lastybin = h_sig.GetYaxis().FindBin(mA+window)

rateSig, rateFake, rateConv, rateVV, rateTTX, rateRare = getRates(firstxbin, lastxbin, firstybin, lastybin)
rateBkg = rateFake+rateConv+rateVV+rateTTX+rateRare
testStat = getTestStat(rateSig, rateBkg)
print(f"rateSig: {rateSig}")
print(f"rateFake: {rateFake}")
print(f"rateConv: {rateConv}")
print(f"rateVV: {rateVV}")
print(f"rateTTX: {rateTTX}")
print(f"rateRare: {rateRare}")
print(f"rateBkg: {rateBkg}")
print(f"testStat: {testStat}")

print(getRatesBySample("ttZToLLNuNu", firstxbin, lastxbin, firstybin, lastybin))

rateSig: 112.76053724050551
rateFake: 291.0
rateConv: -1.0623710635419568
rateVV: 61.90766792041143
rateTTX: 103.98456662318898
rateRare: 3.641971476945029
rateBkg: 459.47183495700347
testStat: 5.064755883934715
74.10264628868266
